# Loading And Validating Data

In [ ]:
import numpy as np
import pandas as pd

DATA = "../Data/Processed"
ID_COL = "Customer ID"

train = pd.read_csv(f"{DATA}/train.csv")
test = pd.read_csv(f"{DATA}/test.csv")
train["split"] = "train"
test["split"] = "test"

full = pd.concat([train, test], ignore_index=True)
eff = pd.read_csv(f"{DATA}/effects_forestdr.csv")

# checks before merging
assert full[ID_COL].is_unique, "duplicate Customer IDs in train+test"
assert eff[ID_COL].is_unique, "duplicate Customer IDs in effects file"
assert len(full) == len(eff), f"row mismatch: {len(full)} vs {len(eff)}"
assert set(full[ID_COL]) == set(eff[ID_COL]), "IDs differ between data and effects"

df = full.merge(eff, on=ID_COL, how="left", validate="one_to_one")

# checks after merging
assert len(df) == len(full)
assert df[["ite_mens", "ite_womens"]].notna().all().all(), "NaNs after merge"

print("shape:", df.shape)
print(df[["ite_mens", "ite_womens"]].describe().round(4))

shape: (64000, 16)
         ite_mens  ite_womens
count  64000.0000  64000.0000
mean       0.0068      0.0032
std        0.0033      0.0035
min       -0.0030     -0.0106
25%        0.0044      0.0009
50%        0.0064      0.0030
75%        0.0089      0.0053
max        0.0214      0.0159


# Checking the effects against the audit

In [43]:
audit = pd.read_csv("../Reports/causal_audit/audit_summary.csv").set_index("treatment")

CONTROL = "No E-Mail"
ARMS = {"Mens E-Mail": "ite_mens", "Womens E-Mail": "ite_womens"}

conv = df.groupby("segment")["conversion"].mean()
rows = []

for arm, col in ARMS.items():
    a = audit.loc[arm]
    raw_ate = conv[arm] - conv[CONTROL]
    lo, hi = a["raw_CI_low_pp"] / 100, a["raw_CI_high_pp"] / 100
    mean_ite = df[col].mean()

    # 1. merged data reproduces the audit 
    assert abs(raw_ate - a["raw_ATE"]) < 1e-5, f"{arm}: recomputed ATE differs from audit"
    # 2. mean predicted effect sits inside the audit confidence interval
    assert lo <= mean_ite <= hi, f"{arm}: mean ITE {mean_ite:.4f} outside audit CI ({lo:.4f}, {hi:.4f})"
    # 4. no absurd values
    assert df[col].abs().max() < 0.05, f"{arm}: effect larger than 5 pp"

    rows.append({
        "arm": arm,
        "audit ATE (pp)": a["raw_ATE"] * 100,
        "recomputed ATE (pp)": raw_ate * 100,
        "audit CI low (pp)": lo * 100,
        "audit CI high (pp)": hi * 100,
        "mean ITE all (pp)": mean_ite * 100,
        "mean ITE train (pp)": df.loc[df["split"] == "train", col].mean() * 100,
        "mean ITE test (pp)": df.loc[df["split"] == "test", col].mean() * 100,
        "ITE std (pp)": df[col].std() * 100,
        "min ITE (pp)": df[col].min() * 100,
        "max ITE (pp)": df[col].max() * 100,
        "share ITE < 0": (df[col] < 0).mean(),
    })

check = pd.DataFrame(rows).round(3).set_index("arm").T
display(check)
print("Step 2 passed: effects are consistent with the audit.")

arm,Mens E-Mail,Womens E-Mail
audit ATE (pp),0.681,0.311
recomputed ATE (pp),0.681,0.311
audit CI low (pp),0.500,0.150
audit CI high (pp),0.861,0.472
mean ITE all (pp),0.682,0.317
mean ITE train (pp),0.683,0.317
mean ITE test (pp),0.678,0.318
ITE std (pp),0.332,0.346
min ITE (pp),-0.296,-1.062
max ITE (pp),2.139,1.594


Step 2 passed: effects are consistent with the audit.


# Setting the revenue value of a conversion

In [44]:
# 1-2. train converters only
conv_train = df[(df["split"] == "train") & (df["conversion"] == 1)]
assert (conv_train["spend"] > 0).all(), "some converters have spend <= 0"

# 3. value of one conversion
VALUE_PER_CONV = conv_train["spend"].mean()

print(f"converters in train : {len(conv_train)}")
print(f"mean spend (USED)   : {VALUE_PER_CONV:.2f}")
print(f"median spend        : {conv_train['spend'].median():.2f}")

# 4. per-arm check only
print("\nPer-arm check (not used):")
print(conv_train.groupby("segment")["spend"].agg(["count", "mean", "median"]).round(2))

# 5. expected incremental revenue per customer
df["rev_mens"] = df["ite_mens"] * VALUE_PER_CONV
df["rev_womens"] = df["ite_womens"] * VALUE_PER_CONV

# 6. sanity check
print("\nExpected revenue gain per e-mail sent (break-even cost):")
print(df[["rev_mens", "rev_womens"]].mean().round(4))
print("\nShare of customers with negative revenue effect:")
print((df[["rev_mens", "rev_womens"]] < 0).mean().round(3))

assert np.isclose(df["rev_mens"].mean(), df["ite_mens"].mean() * VALUE_PER_CONV)
assert np.isclose(df["rev_womens"].mean(), df["ite_womens"].mean() * VALUE_PER_CONV)
print("\nStep 4 passed.")

converters in train : 463
mean spend (USED)   : 113.83
median spend        : 76.16

Per-arm check (not used):
               count    mean  median
segment                             
Mens E-Mail      214  115.39   66.30
No E-Mail         98  104.97   80.97
Womens E-Mail    151  117.37   87.82

Expected revenue gain per e-mail sent (break-even cost):
rev_mens      0.7762
rev_womens    0.3611
dtype: float64

Share of customers with negative revenue effect:
rev_mens      0.005
rev_womens    0.168
dtype: float64

Step 4 passed.


# Setting the email costs and the budget

In [46]:
# 1. PARAMETERS: replace the costs with your real cost per e-mail
COST = {"Mens E-Mail": 0.15, "Womens E-Mail": 0.10}   # dollars per e-mail sent
BUDGET_SHARES = [0.10, 0.25, 0.50]
MAIN_SHARE = 0.25

assert all(c > 0 for c in COST.values()), "costs must be positive"

n = len(df)
be = {"Mens E-Mail": df["rev_mens"].mean(), "Womens E-Mail": df["rev_womens"].mean()}

print("Cost vs break-even (average extra revenue per e-mail):")
for arm in COST:
    flag = "OK" if COST[arm] < be[arm] else "ABOVE BREAK-EVEN"
    print(f"  {arm:14s} cost {COST[arm]:.3f}  break-even {be[arm]:.3f}  -> {flag}")

# 2. unconstrained spend: best arm per customer, only if net value > 0
net_m = df["rev_mens"] - COST["Mens E-Mail"]
net_w = df["rev_womens"] - COST["Womens E-Mail"]
best_is_mens = net_m >= net_w
best_net = np.maximum(net_m, net_w)
send = best_net > 0
unc_cost = np.where(best_is_mens, COST["Mens E-Mail"], COST["Womens E-Mail"])[send].sum()
print(f"\nUnconstrained: {send.sum():,} of {n:,} customers worth e-mailing, "
      f"spend ${unc_cost:,.0f}")

# 3-4. budget scenarios and binding check
full_cost = n * min(COST.values())
rows = []
for s in BUDGET_SHARES:
    B = s * full_cost
    rows.append({
        "share of full cost": s,
        "budget B ($)": round(B, 2),
        "max e-mails (cheapest arm)": int(B // min(COST.values())),
        "binds?": B < unc_cost,
    })
budgets = pd.DataFrame(rows)
display(budgets)
assert budgets.loc[budgets["share of full cost"] == MAIN_SHARE, "binds?"].item(), \
    "main budget does not bind: lower MAIN_SHARE or raise the costs"

# 5. one parameter block, saved in step 7
PARAMS = {
    "COST": COST,
    "VALUE_PER_CONV": VALUE_PER_CONV,
    "BUDGET": MAIN_SHARE * full_cost,
    "BUDGET_SHARES": BUDGET_SHARES,
}
print("\nBudget used in the main run:", round(PARAMS["BUDGET"], 2))
print("Step 5 passed.")

Cost vs break-even (average extra revenue per e-mail):
  Mens E-Mail    cost 0.150  break-even 0.776  -> OK
  Womens E-Mail  cost 0.100  break-even 0.361  -> OK

Unconstrained: 63,443 of 64,000 customers worth e-mailing, spend $8,821


,share of full cost,budget B ($),max e-mails (cheapest arm),binds?
0,0.10,640.0,6399,True
1,0.25,1600.0,15999,True
2,0.50,3200.0,31999,True



Budget used in the main run: 1600.0
Step 5 passed.
